# Module 03, Notebook 2: Placebo Tests and Permutation Inference

## Learning Objectives
By completing this notebook, you will:
1. Implement in-space placebo tests by reassigning treatment to each donor
2. Build the spaghetti plot showing all placebo gaps alongside the treated unit
3. Compute the permutation p-value using the RMSPE ratio statistic
4. Understand why poor-fit donors are excluded from inference
5. Run an in-time placebo test to validate the pre-period fit

## The Inference Problem

Classical inference requires a sampling distribution. With one treated unit, there is no
classical sampling distribution. Permutation tests solve this by asking:

> "If we apply the same procedure to each donor unit (pretending it was treated), do we get gaps
> as large as California's?"

If California's gap is unusually large relative to all donor placebos, we reject the null hypothesis
that the tobacco tax had no effect.

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["Implement in-space placebo tests by reassigning treatment to each donor", "Build the spaghetti plot showing all placebo gaps alongside the treated unit", "Compute the permutation p-value using the RMSPE ratio statistic", "Understand why poor-fit donors are excluded from inference", "Run an in-time placebo test to validate the pre-period fit"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

In [ ]:
# Recreate the panel dataset from Notebook 01

N_YEARS = 30
N_PRE = 20
N_POST = 10
N_DONORS = 20
T_STAR = N_PRE
TRUE_LEVEL_EFFECT = -20.0
TRUE_SLOPE_EFFECT = -0.5

years = np.arange(N_YEARS)

np.random.seed(42)
donor_baselines = np.random.uniform(100, 140, N_DONORS)
shared_trend = -1.2
donor_own_trends = np.random.normal(0, 0.3, N_DONORS)

Y_donors = np.zeros((N_YEARS, N_DONORS))
for j in range(N_DONORS):
    trend = shared_trend + donor_own_trends[j]
    Y_donors[:, j] = (
        donor_baselines[j] + trend * years
        + np.random.normal(0, 3.0, N_YEARS)
    )

TRUE_WEIGHTS = np.zeros(N_DONORS)
TRUE_WEIGHTS[0] = 0.40
TRUE_WEIGHTS[3] = 0.35
TRUE_WEIGHTS[7] = 0.25

y_california_latent = Y_donors @ TRUE_WEIGHTS + np.random.normal(0, 1.5, N_YEARS)
y_california = y_california_latent.copy()
for t in range(T_STAR, N_YEARS):
    time_post = t - T_STAR
    y_california[t] += TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * time_post

# Combine into a single outcome matrix: column 0 = California, columns 1-20 = donors
Y_all = np.column_stack([y_california, Y_donors])  # shape (30, 21)
n_units = Y_all.shape[1]  # 21 units total
TREATED_IDX = 0            # California is unit 0

print(f"Y_all shape: {Y_all.shape}  (years x units)")
print(f"Treated unit index: {TREATED_IDX} (California)")

## 1. Core Functions

Build the weight optimization and placebo test functions.

In [ ]:
def sc_weights(y_treated_pre, Y_donors_pre):
    """Solve for synthetic control weights via constrained quadratic optimization."""
    n_donors = Y_donors_pre.shape[1]
    w0 = np.ones(n_donors) / n_donors

    def obj(w):
        return np.sum((y_treated_pre - Y_donors_pre @ w) ** 2)

    result = minimize(
        obj, w0,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_donors,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
        options={"maxiter": 2000, "ftol": 1e-12},
    )
    return result.x


def run_all_placebos(Y_all, n_pre):
    """
    Run in-space placebo tests for every unit.

    For each unit j:
    - Treat unit j as the "treated" unit
    - All other units are donors
    - Solve for weights and compute pre- and post-period gaps

    Returns
    -------
    gaps_post : array (n_post, n_units) — post-period gap for each unit
    gaps_pre  : array (n_pre, n_units) — pre-period residual for each unit
    rmspe_pre : array (n_units,) — pre-period RMSPE for each unit
    """
    n_periods, n_units = Y_all.shape
    n_post = n_periods - n_pre

    gaps_post = np.zeros((n_post, n_units))
    gaps_pre = np.zeros((n_pre, n_units))
    rmspe_pre = np.zeros(n_units)

    for j in range(n_units):
        y_j_pre = Y_all[:n_pre, j]
        y_j_post = Y_all[n_pre:, j]

        donor_cols = [k for k in range(n_units) if k != j]
        Y_d_pre = Y_all[:n_pre, :][:, donor_cols]
        Y_d_post = Y_all[n_pre:, :][:, donor_cols]

        w = sc_weights(y_j_pre, Y_d_pre)

        y_syn_pre = Y_d_pre @ w
        y_syn_post = Y_d_post @ w

        gaps_pre[:, j] = y_j_pre - y_syn_pre
        gaps_post[:, j] = y_j_post - y_syn_post
        rmspe_pre[j] = np.sqrt(np.mean((y_j_pre - y_syn_pre) ** 2))

    return gaps_post, gaps_pre, rmspe_pre


print("Running placebo tests for all 21 units (this takes ~30 seconds)...")
gaps_post, gaps_pre, rmspe_pre = run_all_placebos(Y_all, N_PRE)
print("Done.")
print(f"gaps_post shape: {gaps_post.shape}  (post_years x units)")

## 2. The Spaghetti Plot

Plot all placebo gaps (gray) alongside California's gap (blue). The treated unit should
stand out clearly in the post-period.

In [ ]:
# Identify poor-fit donors (pre-period RMSPE > 2x California's)
ca_rmspe_pre = rmspe_pre[TREATED_IDX]
good_fit_mask = rmspe_pre <= 2.0 * ca_rmspe_pre

print(f"California pre-period RMSPE: {ca_rmspe_pre:.2f}")
print(f"2x threshold:                {2 * ca_rmspe_pre:.2f}")
print(f"Good-fit donors:             {good_fit_mask.sum() - 1} (excluding California)")
print(f"Poor-fit donors (excluded):  {(~good_fit_mask).sum()}")

# Spaghetti plot
post_years = years[N_PRE:]
all_years = years

fig, ax = plt.subplots(figsize=(13, 6))

# Pre-period gaps for all units
for j in range(n_units):
    if j == TREATED_IDX:
        continue
    alpha = 0.35 if good_fit_mask[j] else 0.12
    color = "#aaaaaa" if good_fit_mask[j] else "#dddddd"
    # Full gap trajectory (pre + post)
    full_gap = np.concatenate([gaps_pre[:, j], gaps_post[:, j]])
    ax.plot(all_years, full_gap, color=color, alpha=alpha, linewidth=1)

# California full gap trajectory
ca_gap_full = np.concatenate([gaps_pre[:, TREATED_IDX], gaps_post[:, TREATED_IDX]])
ax.plot(all_years, ca_gap_full, color="#2980b9", linewidth=2.5,
        zorder=5, label="California (treated)")

ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2, label="Tobacco tax")

ax.set_title("Placebo Test: California vs All Donor Placebos", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Gap (Treated − Synthetic) [packs per capita]")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Pre-period (years 0-19): All lines near zero confirms good pre-period fit")
print("  Post-period (years 20-29): California clearly below all donor placebos")

## 3. Compute the Permutation P-Value

The RMSPE ratio statistic: post-period RMSPE divided by pre-period RMSPE.
This controls for units with poor pre-period fit (which would otherwise inflate the null distribution).

In [ ]:
# Compute RMSPE ratio for all units
rmspe_post = np.sqrt(np.mean(gaps_post ** 2, axis=0))
rmspe_ratio = rmspe_post / rmspe_pre

# Full permutation p-value (all units)
ca_ratio = rmspe_ratio[TREATED_IDX]
p_value_full = (rmspe_ratio >= ca_ratio).mean()

# Filtered p-value (exclude poor-fit donors)
rmspe_ratio_filtered = rmspe_ratio[good_fit_mask]
p_value_filtered = (rmspe_ratio_filtered >= ca_ratio).mean()

min_p_value = 1.0 / n_units
min_p_filtered = 1.0 / good_fit_mask.sum()

print("Permutation P-Values")
print("=" * 50)
print(f"  California RMSPE ratio:        {ca_ratio:.3f}")
print()
print(f"  All units ({n_units} total):")
print(f"    P-value:               {p_value_full:.4f}")
print(f"    Minimum achievable:    {min_p_value:.4f}")
print()
print(f"  Good-fit units ({good_fit_mask.sum()} total):")
print(f"    P-value:               {p_value_filtered:.4f}")
print(f"    Minimum achievable:    {min_p_filtered:.4f}")
print()

if p_value_filtered < 0.05:
    print("  SIGNIFICANT at 5%: California's post-period gap is unusually large.")
elif p_value_filtered < 0.10:
    print("  MARGINALLY significant at 10%.")
else:
    print("  NOT significant at 10%: California's gap is within the placebo distribution.")

# Show the rank of California
ca_rank = (rmspe_ratio >= ca_ratio).sum()  # how many units have ratio >= California
print(f"\n  California's rank: {ca_rank} out of {n_units} units")
print(f"  (rank 1 = largest ratio = most significant)")

In [ ]:
# Visualize RMSPE ratio distribution

fig, ax = plt.subplots(figsize=(10, 5))

# Bar chart of RMSPE ratios: donors in gray, California in blue
colors = ["#2980b9" if j == TREATED_IDX else "#aaaaaa" for j in range(n_units)]
labels = ["California" if j == TREATED_IDX else f"Donor {j}" for j in range(n_units)]

sorted_idx = np.argsort(rmspe_ratio)[::-1]
bar_colors = [colors[i] for i in sorted_idx]
bar_values = rmspe_ratio[sorted_idx]

ax.bar(range(n_units), bar_values, color=bar_colors, alpha=0.85, edgecolor="white")
ax.set_title("RMSPE Ratio: Post-Period Gap / Pre-Period Fit\n(Blue = California, Gray = Donors)",
             fontsize=12)
ax.set_xlabel("Units (sorted by RMSPE ratio)")
ax.set_ylabel("RMSPE Post / RMSPE Pre")

# Mark California explicitly
ca_sorted_pos = np.where(sorted_idx == TREATED_IDX)[0][0]
ax.annotate("California", xy=(ca_sorted_pos, ca_ratio),
            xytext=(ca_sorted_pos + 1, ca_ratio + 0.3),
            fontsize=10, color="#2980b9",
            arrowprops=dict(arrowstyle="->", color="#2980b9"))

plt.tight_layout()
plt.show()

print(f"Permutation p-value = {p_value_filtered:.4f}")
print("California's RMSPE ratio should be the largest (leftmost blue bar).")

## 4. In-Time Placebo Test

A complementary validation: pretend the intervention happened in year 14 (6 years before the
true intervention at year 20). The synthetic control should show a near-zero gap in this
placebo post-period (years 14–19), since there was no real intervention then.

In [ ]:
# In-time placebo: pretend intervention at year 14
PLACEBO_T0 = 14

# Use only years 0-13 for weight optimization
y_ca_placebo_pre = y_california[:PLACEBO_T0]
Y_donors_placebo_pre = Y_donors[:PLACEBO_T0, :]
Y_donors_placebo_post = Y_donors[PLACEBO_T0:N_PRE, :]  # years 14-19 (placebo post)

w_placebo = sc_weights(y_ca_placebo_pre, Y_donors_placebo_pre)

y_syn_placebo_pre = Y_donors_placebo_pre @ w_placebo
y_syn_placebo_post = Y_donors_placebo_post @ w_placebo

gap_placebo_pre = y_ca_placebo_pre - y_syn_placebo_pre
gap_placebo_post = y_california[PLACEBO_T0:N_PRE] - y_syn_placebo_post

rmspe_placebo_pre = np.sqrt(np.mean(gap_placebo_pre ** 2))
rmspe_placebo_post = np.sqrt(np.mean(gap_placebo_post ** 2))

print("In-Time Placebo Test (placebo intervention at year 14)")
print(f"  Pre-placebo RMSPE (years 0-13):   {rmspe_placebo_pre:.2f}")
print(f"  Placebo post RMSPE (years 14-19): {rmspe_placebo_post:.2f}")
print(f"  RMSPE ratio:                      {rmspe_placebo_post / rmspe_placebo_pre:.2f}")
print()

# Compare to true analysis RMSPE ratio
print(f"  True analysis RMSPE ratio (California): {ca_ratio:.2f}")
if rmspe_placebo_post / rmspe_placebo_pre < 0.5 * ca_ratio:
    print("  IN-TIME PLACEBO PASSES: Placebo ratio much smaller than true analysis ratio.")
    print("  The true intervention creates a gap distinctly larger than any pre-period gap.")
else:
    print("  IN-TIME PLACEBO FAILS: Inspect pre-period fit — potential pre-period violation.")

# Visualize the in-time placebo
fig, ax = plt.subplots(figsize=(10, 5))

placebo_years = years[PLACEBO_T0:N_PRE]
ax.plot(years[:PLACEBO_T0], gap_placebo_pre, color="#aaaaaa", linewidth=1.5,
        label="Pre-placebo gap")
ax.plot(placebo_years, gap_placebo_post, color="#e67e22", linewidth=2.5,
        label="Placebo post gap (years 14-19)")
ax.plot(years[N_PRE:], gaps_post[:, TREATED_IDX], color="#2980b9", linewidth=2.5,
        label="True post gap (years 20-29)")
ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(PLACEBO_T0, color="#e67e22", linestyle=":", linewidth=2, label="Placebo date (Year 14)")
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2, label="True intervention (Year 20)")
ax.set_title("In-Time Placebo Test: Year 14 vs True Intervention at Year 20", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Gap (packs per capita)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Summary

### Inference Results

| Test | Result |
|------|--------|
| In-space permutation p-value | See output above |
| In-time placebo (year 14) | Near-zero gap — validates pre-period |
| California's RMSPE rank | 1st (largest gap relative to fit quality) |

### Key Concepts

1. **Permutation inference**: the null distribution comes from applying the same procedure to each donor (treating it as if it were California). No distributional assumptions required.

2. **RMSPE ratio**: normalizing the post-period gap by the pre-period fit prevents poor-fit donors from inflating the null distribution.

3. **In-time placebo**: validates the approach by checking there is no spurious large gap at a pre-period placebo date.

4. **Minimum p-value**: with 21 units, the minimum achievable p-value is 1/21 ≈ 0.048 — just barely significant at 5% if California has the largest ratio.

### What's Next

**Notebook 03:** CausalPy Synthetic Control — use `cp.SyntheticControl` for a full Bayesian
analysis that adds posterior uncertainty quantification on top of the classical synthetic control.

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])